# Loading 

In [22]:
import pandas as pd
import geopandas as gpd
from shapely import wkt

In [23]:
# Precinct shapes with TownshipName already attached
dupage_precincts = gpd.read_file('Election_Precincts.geojson')

# Your cleaned Census data
df_muslim_dupage = pd.read_csv('df_muslim_clean_dupage.csv')
df_citizen_dupage = pd.read_csv('df_citizen_dupage_clean.csv')

print(dupage_precincts.shape)
print(df_muslim_dupage.shape)
print(df_citizen_dupage.shape)

(600, 15)
(219, 22)
(219, 3)


In [24]:
print(dupage_precincts.columns.tolist())
print(dupage_precincts['TownshipName'].value_counts())

['OBJECTID', 'Precinct', 'TownshipName', 'TownshipCode', 'PrecinctName', 'PrecinctID', 'Label', 'PollingID', 'Format', 'LastEditor', 'LastUpdate', 'Notes', 'ShapeSTArea', 'ShapeSTLength', 'geometry']
TownshipName
Downers Grove    103
York              86
Milton            81
Lisle             77
Bloomingdale      72
Naperville        58
Addison           53
Wayne             43
Winfield          27
Name: count, dtype: int64


# Polygons

In [25]:
# Precinct shapes with TownshipName already attached
dupage_precincts = gpd.read_file('Election_Precincts.geojson')

# Your cleaned Census data
df_muslim_dupage = pd.read_csv('df_muslim_clean_dupage.csv')
df_citizen_dupage = pd.read_csv('df_citizen_dupage_clean.csv')

print(dupage_precincts.shape)
print(df_muslim_dupage.shape)
print(df_citizen_dupage.shape)

(600, 15)
(219, 22)
(219, 3)


In [26]:
print(dupage_precincts.columns.tolist())
print(dupage_precincts['TownshipName'].value_counts())

['OBJECTID', 'Precinct', 'TownshipName', 'TownshipCode', 'PrecinctName', 'PrecinctID', 'Label', 'PollingID', 'Format', 'LastEditor', 'LastUpdate', 'Notes', 'ShapeSTArea', 'ShapeSTLength', 'geometry']
TownshipName
Downers Grove    103
York              86
Milton            81
Lisle             77
Bloomingdale      72
Naperville        58
Addison           53
Wayne             43
Winfield          27
Name: count, dtype: int64


In [27]:
dupage_townships_gdf = dupage_precincts.dissolve(by='TownshipName', as_index=False)
dupage_townships_gdf = dupage_townships_gdf[['TownshipName', 'geometry']]
print(dupage_townships_gdf.shape)  

(9, 2)


In [28]:
dupage_tracts_gdf = gpd.read_file('tl_2023_17_tract.shp')
dupage_tracts_gdf = dupage_tracts_gdf[dupage_tracts_gdf['COUNTYFP'] == '043']
print(dupage_tracts_gdf.shape) 

(219, 14)


In [29]:
dupage_tracts_gdf['GEOID'] = dupage_tracts_gdf['GEOID'].astype(str)
df_muslim_dupage['GEO_ID'] = df_muslim_dupage['GEO_ID'].astype(str)
df_citizen_dupage['GEO_ID'] = df_citizen_dupage['GEO_ID'].astype(str)

dupage_tracts_gdf = dupage_tracts_gdf.merge(df_muslim_dupage, left_on='GEOID', right_on='GEO_ID', how='inner')
dupage_tracts_gdf = dupage_tracts_gdf.merge(df_citizen_dupage, on='GEO_ID', how='left')

print(dupage_tracts_gdf.shape)
print(dupage_tracts_gdf['muslim_foreign_born'].sum())      # should be 15606
print(dupage_tracts_gdf['naturalized_citizens'].sum())     # should be 108871
print(dupage_tracts_gdf['non_citizens'].sum())             # should be 75962

(219, 38)
15606
108871
75962


# Spatial Join & Merge

In [30]:
# Match CRS between tracts and townships
dupage_tracts_gdf = dupage_tracts_gdf.to_crs(dupage_townships_gdf.crs)

# Centroid each tract
dupage_tract_centroids = dupage_tracts_gdf.copy()
dupage_tract_centroids['geometry'] = dupage_tract_centroids.geometry.centroid

# Spatial join where township polygon contains each tract centroid
dupage_tract_to_township = gpd.sjoin(dupage_tract_centroids, dupage_townships_gdf, how='left', predicate='within')

print(dupage_tract_to_township.shape)   
print(dupage_tract_to_township['TownshipName'].isna().sum())  

(219, 40)
0


C:\Users\abdif\AppData\Local\Temp\ipykernel_23192\373402109.py:6: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  dupage_tract_centroids['geometry'] = dupage_tract_centroids.geometry.centroid


In [31]:
dupage_township_pop = dupage_tract_to_township.groupby('TownshipName')[['muslim_foreign_born', 'naturalized_citizens', 'non_citizens']].sum().reset_index()

print(dupage_township_pop.shape)  
print(dupage_township_pop)

# Sanity check: totals should still match known DuPage-wide numbers
print(dupage_township_pop['muslim_foreign_born'].sum())      
print(dupage_township_pop['naturalized_citizens'].sum())     
print(dupage_township_pop['non_citizens'].sum())             

(9, 4)
    TownshipName  muslim_foreign_born  naturalized_citizens  non_citizens
0        Addison                  707                 13916          9977
1   Bloomingdale                 2878                 18450         10187
2  Downers Grove                 2809                 15907          9828
3          Lisle                 1588                 11410          9791
4         Milton                 2804                 11058          6406
5     Naperville                 1222                 11705         12879
6          Wayne                 1260                  9858          3837
7       Winfield                  361                  4639          5689
8           York                 1977                 11928          7368
15606
108871
75962


In [32]:
dupage_election = pd.read_csv('dupage_township_election_2022_general.csv')

dupage_full = dupage_township_pop.merge(dupage_election, left_on='TownshipName', right_on='township', how='left')
print(dupage_full.shape) 
print(dupage_full)

(9, 8)
    TownshipName  muslim_foreign_born  naturalized_citizens  non_citizens  \
0        Addison                  707                 13916          9977   
1   Bloomingdale                 2878                 18450         10187   
2  Downers Grove                 2809                 15907          9828   
3          Lisle                 1588                 11410          9791   
4         Milton                 2804                 11058          6406   
5     Naperville                 1222                 11705         12879   
6          Wayne                 1260                  9858          3837   
7       Winfield                  361                  4639          5689   
8           York                 1977                 11928          7368   

        township  registered  ballots_cast    turnout  
0        Addison       50373         23874  47.394437  
1   Bloomingdale       69894         32750  46.856669  
2  Downers Grove      106273         64204  60.414216 

In [33]:
dupage_full.to_csv('dupage_township_pop.csv', index=False)
print('saved:', dupage_full.shape)

saved: (9, 8)


# Manual Check

In [34]:
bloomingdale_tract = dupage_tract_to_township[dupage_tract_to_township['TownshipName'] == 'Bloomingdale'].iloc[0]
winfield_tract = dupage_tract_to_township[dupage_tract_to_township['TownshipName'] == 'Winfield'].iloc[0]

print(f"Bloomingdale: GEOID={bloomingdale_tract['GEOID']}, muslim_foreign_born={bloomingdale_tract['muslim_foreign_born']}, centroid=({bloomingdale_tract.geometry.centroid.y}, {bloomingdale_tract.geometry.centroid.x})")
print(f"Winfield: GEOID={winfield_tract['GEOID']}, muslim_foreign_born={winfield_tract['muslim_foreign_born']}, centroid=({winfield_tract.geometry.centroid.y}, {winfield_tract.geometry.centroid.x})")

Bloomingdale: GEOID=17043841205, muslim_foreign_born=25, centroid=(41.923488665060574, -88.13338509955297)
Winfield: GEOID=17043841603, muslim_foreign_born=137, centroid=(41.840238927597184, -88.173159982368)


In [35]:
for geoid in ['17043841205', '17043841603']:
    raw_muslim = df_muslim_dupage[df_muslim_dupage['GEO_ID'] == geoid]['muslim_foreign_born'].values
    print(f"{geoid}: raw_muslim={raw_muslim}")

17043841205: raw_muslim=[25]
17043841603: raw_muslim=[137]


In [36]:
bloomingdale_manual = dupage_tract_to_township[dupage_tract_to_township['TownshipName'] == 'Bloomingdale']['muslim_foreign_born'].sum()
bloomingdale_pipeline = dupage_township_pop[dupage_township_pop['TownshipName'] == 'Bloomingdale']['muslim_foreign_born'].values[0]
print(f"Bloomingdale: manual={bloomingdale_manual}, pipeline={bloomingdale_pipeline}, match={bloomingdale_manual == bloomingdale_pipeline}")

winfield_manual = dupage_tract_to_township[dupage_tract_to_township['TownshipName'] == 'Winfield']['muslim_foreign_born'].sum()
winfield_pipeline = dupage_township_pop[dupage_township_pop['TownshipName'] == 'Winfield']['muslim_foreign_born'].values[0]
print(f"Winfield: manual={winfield_manual}, pipeline={winfield_pipeline}, match={winfield_manual == winfield_pipeline}")

Bloomingdale: manual=2878, pipeline=2878, match=True
Winfield: manual=361, pipeline=361, match=True


In [37]:
dupage_townships_gdf['geometry'] = dupage_townships_gdf['geometry'].apply(lambda geom: geom.wkt)
dupage_townships_gdf.to_csv('dupage_shapes.csv', index=False)
print(dupage_townships_gdf.shape)

(9, 2)


C:\Users\abdif\AppData\Local\Temp\ipykernel_23192\1482070414.py:1: UserWarning: Geometry column does not contain geometry.
  dupage_townships_gdf['geometry'] = dupage_townships_gdf['geometry'].apply(lambda geom: geom.wkt)
